# Stage 1 profiling
Seven signals (ACT, LOY, PLAN, PAY, ENG, OFFER, CONS), five routes, nine candidates, the same 8 checks on each, then a weighted score.

In [1]:
# 1 setup + 8-check profile of every fetched candidate
import sys, pathlib; ROOT = pathlib.Path.cwd().parent; sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data.profile_data import build_profile_summary
summary = build_profile_summary(ROOT / "data" / "profile_summary.csv")
summary[["dataset_id","name","rows","cols","pct_missing","dup_rows","label_balance","sensitive_cols"]]

profiling D1 IBM Telco Customer Churn
skip D2 (not fetched)
profiling D3 Orange Telecom (BigML)
profiling D4 Orange Belgium uplift


profiling D5 UCI Online Retail II


skip D6 (not fetched)
profiling D7 UCI Default of Credit Card Clients


skip D8 (not fetched)
profiling R3 Scraped public offers
wrote C:\Users\hp\Desktop\loyalty_assistant\data\profile_summary.csv


,dataset_id,name,rows,cols,pct_missing,dup_rows,label_balance,sensitive_cols
0,D1,IBM Telco Customer Churn,7043,21,0.007,0,"{""No"": 0.735, ""Yes"": 0.265}",gender;SeniorCitizen;Partner;Dependents;PhoneS...
1,D3,Orange Telecom (BigML),5000,21,0.000,0,"{""0"": 0.859, ""1"": 0.141}",state;area_code;phone_number
2,D4,Orange Belgium uplift,11896,180,0.000,0,"{""0"": 0.966, ""1"": 0.034}",
3,D5,UCI Online Retail II,1067371,8,2.897,34335,NaN,Country
4,D7,UCI Default of Credit Card Clients,30000,25,0.000,0,"{""0"": 0.779, ""1"": 0.221}",SEX;MARRIAGE;AGE
5,R3,Scraped public offers,15,9,11.111,0,"{""hand_written"": 1.0}",


In [2]:
# 2 telco deep-dive
telco = pd.read_csv(ROOT/"data/raw/telco/telco_churn.csv")
print((telco.TotalCharges.astype(str).str.strip()=="").sum(), "blank TotalCharges (all tenure 0)")
telco.Churn.value_counts(normalize=True).round(3)

11 blank TotalCharges (all tenure 0)


Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64

In [3]:
# 3 credit: late share (target for S2-T07 fusion sanity)
credit = pd.read_csv(ROOT/"data/raw/credit/credit_default.csv")
pay = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
print("share of clients with any late month:", (credit[pay] >= 1).any(axis=1).mean().round(3))

share of clients with any late month: 0.336


In [4]:
# 4 scoring: 1-5 per criterion, total = weighted sum / 5 (out of 100)
W = {"coverage":30,"labels":15,"size_quality":15,"domain_fit":15,"licence":10,"effort":15}
rows = [
  ["D1","IBM Telco",            3,5,4,5,4,5], ["D2","Maven Telecom",      4,5,4,5,4,2],
  ["D3","Orange BigML",         2,5,3,4,2,4], ["D4","Orange Belgium",     2,4,4,3,1,3],
  ["D5","UCI Online Retail II", 2,1,5,3,5,3], ["D6","Olist",              3,2,4,2,1,2],
  ["D7","UCI Credit Default",   2,5,5,2,5,4], ["D8","TRAI reports",       1,1,2,3,4,2],
  ["R3","Offers (scraped/hand)",1,1,2,5,4,4],
]
scores = pd.DataFrame(rows, columns=["dataset_id","name",*W])
scores["total"] = sum(scores[k]*w for k,w in W.items())/5
scores = scores.sort_values("total", ascending=False)
scores.to_csv(ROOT/"data/candidate_scores.csv", index=False)
scores

,dataset_id,name,coverage,labels,size_quality,domain_fit,licence,effort,total
0,D1,IBM Telco,3,5,4,5,4,5,83.0
1,D2,Maven Telecom,4,5,4,5,4,2,80.0
6,D7,UCI Credit Default,2,5,5,2,5,4,70.0
2,D3,Orange BigML,2,5,3,4,2,4,64.0
4,D5,UCI Online Retail II,2,1,5,3,5,3,58.0
3,D4,Orange Belgium,2,4,4,3,1,3,56.0
5,D6,Olist,3,2,4,2,1,2,50.0
8,R3,Offers (scraped/hand),1,1,2,5,4,4,50.0
7,D8,TRAI reports,1,1,2,3,4,2,38.0
